In [2]:
import torch

a= torch.tensor([[1,2,3],[4,5,6]])
b= torch.zeros(2,3)
c= torch.randn(2,3)
d= torch.arange(6)

for name, t in [("a", a), ("b", b), ("c", c), ("d", d)]:
    print(f"{name}: shape={tuple(t.shape)} dtype={t.dtype} device={t.device}")

a: shape=(2, 3) dtype=torch.int64 device=cpu
b: shape=(2, 3) dtype=torch.float32 device=cpu
c: shape=(2, 3) dtype=torch.float32 device=cpu
d: shape=(6,) dtype=torch.int64 device=cpu


In [10]:
print(torch.tensor(3.0).shape)
print(torch.tensor([3.0]).shape)
print(a.ndim, a.numel())

torch.Size([])
torch.Size([1])
2 6


In [13]:
for dt in [torch.float64, torch.float32, torch.float16, torch.int64, torch.uint8]:
    t = torch.zeros(1000, 1000, dtype=dt)
    mb = t.numel() * t.element_size() / 1024 ** 2
    print(f"{str(dt):<16} element_size={t.element_size()}B   1000x1000 = {mb:6.2f} MiB")

torch.float64    element_size=8B   1000x1000 =   7.63 MiB
torch.float32    element_size=4B   1000x1000 =   3.81 MiB
torch.float16    element_size=2B   1000x1000 =   1.91 MiB
torch.int64      element_size=8B   1000x1000 =   7.63 MiB
torch.uint8      element_size=1B   1000x1000 =   0.95 MiB


In [15]:
x = torch.arange(5)
print(x.dtype, x.float().dtype)
print(x.to(torch.float16).dtype)

y = torch.tensor([1.9, -1.9])
print(y.to(torch.int64))

torch.int64 torch.float32
torch.float16
tensor([ 1, -1])


In [19]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

x_cpu = torch.rand(3,3)
x_gpu = x_cpu.to(dev)
print(x_cpu.device, x_gpu.device)

try:
    x_cpu + x_gpu
except RuntimeError as e:
    print("RuntimeError:", str(e)[:80])

device: cuda
cpu cuda:0
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, c


In [23]:
x = torch.arange(12).reshape(3,4)
print(x)
print("shape:",tuple(x.shape))
print("stride:", x.stride)
print("contiguous:", x.contiguous())

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
shape: (3, 4)
stride: <built-in method stride of Tensor object at 0x000002FA04BDAE40>
contiguous: tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


In [25]:
xt = x.t();
print("shape:",tuple(xt.shape))
print("stride:", xt.stride)
print("contiguous:", xt.contiguous())

print("is same memory?", xt.data_ptr() == x.data_ptr())

shape: (4, 3)
stride: <built-in method stride of Tensor object at 0x000002FA04BC6080>
contiguous: tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
is same memory? True


In [27]:
try:
    xt.view(12)
except RuntimeError as e:
    print("RuntimeError:", str(e)[:120])

print(xt.reshape(12))
print(xt.contiguous().view(12))

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subs
tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])
tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])


In [30]:
img = torch.randn(2,3,32,32)
print(img.permute(0,2,3,1).shape)

v=torch.randn(5)
print(v.unsqueeze(0).shape, v.unsqueeze(1).shape)
print(v.unsqueeze(0).squeeze().shape)

torch.Size([2, 32, 32, 3])
torch.Size([1, 5]) torch.Size([5, 1])
torch.Size([5])


In [33]:
cases = [((3, 1), (1, 4)), ((2, 3, 4), (4,)), ((5,), (5, 1)), ((2, 3), (3, 2))]

for s1, s2 in cases:
    try:
        r = tuple((torch.zeros(*s1) + torch.zeros(*s2)).shape)
        print(f"{str(s1):<12} + {str(s2):<10} = {r}")
    except RuntimeError:
        print(f"{str(s1):<12} + {str(s2):<10} = 에러")

(3, 1)       + (1, 4)     = (3, 4)
(2, 3, 4)    + (4,)       = (2, 3, 4)
(5,)         + (5, 1)     = (5, 5)
(2, 3)       + (3, 2)     = 에러


In [35]:
img = torch.randn(4,3,8,8)
mean = torch.tensor([0.485, 0.456, 0.406])
std  = torch.tensor([0.229, 0.224, 0.225])

try:
    (img - mean)
except RuntimeError as e:
    print("그냥 빼면:", str(e)[:70])

m = mean.view(3,1,1)
out = (img - m) / std.view(3,1,1)
print(out.shape)
print(out[:,0].mean().item(), out[:,1].mean().item())

그냥 빼면: The size of tensor a (8) must match the size of tensor b (3) at non-si
torch.Size([4, 3, 8, 8])
-2.231682777404785 -2.120814085006714


In [37]:
big = torch.zeros(1000,1000)
row = torch.zeros(1000)

print("row 실제 크기 :", row.numel() * row.element_size() / 1024**2, "MiB")
print("expand 후 shape:", tuple(row.expand(1000, 1000).shape))
print("expand 후 stride:", row.expand(1000, 1000).stride())

row 실제 크기 : 0.003814697265625 MiB
expand 후 shape: (1000, 1000)
expand 후 stride: (0, 1)


# step5

In [39]:
import time

n = 4096
x_cpu = torch.randn(n,n)

torch.cuda.synchronize()
t = time.time(); x_gpu = x_cpu.to("cuda"); torch.cuda.synchronize()
print(f"CPU→GPU : {time.time()-t:.4f}s")

torch.cuda.synchronize()
t = time.time(); _ = x_gpu.cpu(); torch.cuda.synchronize()
print(f"GPU→CPU : {time.time()-t:.4f}s")

torch.cuda.synchronize()
t = time.time(); _ = x_gpu @ x_gpu; torch.cuda.synchronize()
print(f"GPU matmul : {time.time()-t:.4f}s")

CPU→GPU : 0.0480s
GPU→CPU : 0.0420s
GPU matmul : 0.1346s


In [41]:
chunks = [torch.randn(512,512) for _ in range(64)]
one = torch.randn(4096,4096)

torch.cuda.synchronize()
t = time.time()
for c in chunks: c.to("cuda")
torch.cuda.synchronize()
print(f"64번 나눠 이동 : {time.time()-t:.4f}s")

torch.cuda.synchronize()
t = time.time(); one.to("cuda"); torch.cuda.synchronize()
print(f"1번에 이동     : {time.time()-t:.4f}s")

64번 나눠 이동 : 0.0440s
1번에 이동     : 0.0400s


In [44]:
torch.cuda.empty_cache()
base = torch.cuda.memory_allocated() / 1024**2
batch = torch.randn(32, 3, 224, 224, device="cuda")     # 흔한 학습 배치 한 개
used = torch.cuda.memory_allocated() / 1024**2 - base

print(f"텐서 1개      : {used:.2f} MiB")
print(f"직접 계산     : {32*3*224*224*4/1024**2:.2f} MiB")
print(f"reserved      : {torch.cuda.memory_reserved()/1024**2:.2f} MiB")
print(f"내 VRAM       : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GiB")

텐서 1개      : 0.00 MiB
직접 계산     : 18.38 MiB
reserved      : 188.00 MiB
내 VRAM       : 16.00 GiB


In [48]:
del batch, x_gpu, one, chunks
torch.cuda.empty_cache()
print(f"allocated: {torch.cuda.memory_allocated()/1024**2:.2f} MiB")

NameError: name 'batch' is not defined

In [51]:
x= torch.randn(8,1,28,28)

print(x.reshape(8,784))

tensor([[ 1.7680,  0.0907,  1.4007,  ...,  1.1307, -0.3541, -0.4112],
        [ 0.3410, -0.3427,  0.8513,  ..., -1.1546, -1.2238, -0.6843],
        [ 0.1557, -0.1570, -0.5581,  ..., -0.9476,  1.2052, -0.1354],
        ...,
        [-1.8011, -0.9395, -0.1783,  ...,  0.3706,  0.4816, -0.2211],
        [-0.2034, -1.2685,  0.1378,  ..., -0.3370, -0.2997,  0.1423],
        [ 0.5828,  0.5364,  1.2228,  ..., -0.5486, -0.5558, -1.6949]])


In [55]:
logits = torch.randn(16,10)

print(logits.shape)

logits.argmax(dim=1)


torch.Size([16, 10])


tensor([1, 8, 1, 5, 2, 1, 5, 4, 6, 9, 7, 8, 3, 7, 9, 9])

In [57]:
a=torch.randn(4,3)
b=torch.randn(4,)

a * b.unsqueeze(1)

tensor([[ 0.5449, -0.8676, -2.5196],
        [-0.0644, -0.6138, -1.0071],
        [-0.5273, -0.4414,  1.9096],
        [-0.1999, -0.1993,  2.2458]])

In [61]:
t = torch.arange(24).reshape(2,3,4)

print(t)

t.permute(2,1,0)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


tensor([[[ 0, 12],
         [ 4, 16],
         [ 8, 20]],

        [[ 1, 13],
         [ 5, 17],
         [ 9, 21]],

        [[ 2, 14],
         [ 6, 18],
         [10, 22]],

        [[ 3, 15],
         [ 7, 19],
         [11, 23]]])

In [69]:
pred = torch.randn(5)

target = torch.randn(5,1)
loss = ((pred - target) ** 2).mean()

print(loss.item(), (pred - target).shape)

0.7169165015220642 torch.Size([5, 5])


In [71]:
x = torch.arange(6.)
y = x[:3]
y += 100
print(x)


tensor([100., 101., 102.,   3.,   4.,   5.])
